**O que este script faz?**

1. **Ingestão de Dados (Extract):**
   Coleta dados de filmes a partir de múltiplas fontes, incluindo arquivos CSV públicos hospedados no GitHub e tabelas analíticas armazenadas no Google BigQuery (GCP).

2. **Transformação e Cruzamento (Transform):**
   Realiza operações de Merge/Join entre datasets de filmes da Marvel e DC, aplica filtros por franquia, ordena por avaliação do IMDB e remove registros duplicados para consolidar uma base analítica unificada.

3. **Engenharia de Strings e Limpeza de Dados:**
   Manipula listas de atores presentes em colunas textuais, realiza sanitização de strings, remove valores nulos e identifica automaticamente o ator principal de cada filme selecionado.

4. **Enriquecimento com APIs Externas:**
   Utiliza a API do TMDB (The Movie Database) para converter nomes de atores em IDs oficiais da plataforma e, posteriormente, recuperar os respectivos usuários públicos do Instagram vinculados aos artistas.

5. **Web Scraping / Coleta de Dados de Redes Sociais:**
   Consome informações públicas do Instagram por meio da biblioteca `igmapper`, coletando métricas de perfil (seguidores, quantidade de posts, seguindo) e estatísticas de publicações (likes, comentários, timestamps e tipos de mídia).

6. **Processamento Analítico de Redes Sociais:**
   Estrutura e consolida os dados sociais em DataFrames do Pandas, organizando informações históricas de posts e métricas agregadas dos perfis dos atores selecionados.

7. **Carga de Dados na Nuvem (Load):**
   Exporta os datasets finais processados para tabelas analíticas no Google BigQuery, utilizando a camada Bronze do Data Lake para armazenamento estruturado dos dados brutos enriquecidos.

8. **Pipeline de Engenharia de Dados End-to-End:**
   Implementa um fluxo completo de ETL/ELT envolvendo ingestão de dados externos, transformação analítica, enriquecimento via APIs, scraping de redes sociais e persistência em ambiente Cloud (GCP).


In [ ]:
# ==============================================================================
# 1. IMPORTAÇÃO DE BIBLIOTECAS E DEPENDÊNCIAS
# ==============================================================================

# Ferramentas de Nuvem (Google Cloud Platform - GCP)
from google.cloud import bigquery  # Conector oficial do banco de dados analítico (BigQuery) da Google.
from google.oauth2 import service_account  # Mecanismo de autenticação por chave/arquivo JSON de conta de serviço.

# Manipulação e Análise de Dados
import pandas as pd  # A principal biblioteca de Data Science para tabelas (DataFrames) e manipulação matricial.

# APIs Externas
import tmdbsimple as tmdb  # Wrapper (facilitador de chamadas) para a API do site "The Movie Database" (TMDB).
import igmapper  # Biblioteca/Scraper personalizado para extração de dados públicos do Instagram.

In [ ]:
# ==============================================================================
# 2. CONFIGURAÇÕES DE VISUALIZAÇÃO (PANDAS DISPLAY CONFIG)
# ==============================================================================

# Por padrão, o Pandas esconde colunas/linhas se a tabela for muito grande.
# Estas configurações garantem que possamos inspecionar os dados no terminal/notebook.
pd.set_option("display.max_columns", None)  # Não limita o número máximo de colunas exibidas na tela.
pd.set_option("display.max_rows", 10)  # Limita a exibição visual a no máximo 10 linhas para não travar a IDE.

In [ ]:
# ==============================================================================
# 3. CONFIGURAÇÃO DE FONTES DE DADOS E INFRAESTRUTURA DE NUVEM
# ==============================================================================

# --- FONTE 1: REPOSITÓRIO REMOTO (GITHUB) ---
# URL base onde estão armazenados arquivos CSV públicos contendo dados históricos de filmes.
url_github_movies_dataset = "https://media.githubusercontent.com/media/lucas-aulas/dataset-movies/refs/heads/main"

# --- FONTE 2: AMBIENTE DE LEITURA (PROJETO COMPARTILHADO DO PROFESSOR) ---
# ID do projeto na Google Cloud de onde leremos uma tabela institucional de bilheterias.
fonte_gcp_project_id = "_______________________"

# --- DESTINO: AMBIENTE DE ESCRITA (PROJETO PRIVADO DO ALUNO) ---
# Caminho local do arquivo JSON que contém as credenciais de segurança (chave privada).
# ATENÇÃO: Nunca envie esta chave para o GitHub público! Ela equivale à senha do seu banco de dados.
meu_service_account_file = "../secrets/_______________________"
meu_gcp_project_id = "_______________________"  # ID do seu projeto pessoal criado no GCP.

# Inicialização da autenticação e do cliente BigQuery
# Transforma o arquivo JSON em um objeto de credenciais aceito pelo Google.
gcp_credentials = service_account.Credentials.from_service_account_file(meu_service_account_file)

# O objeto 'client' funciona como a nossa ponte ativa de conexão/comunicação com a nuvem do Google.
bigquery_client = bigquery.Client(credentials=gcp_credentials)

# Chave da API do IMDB para consulta de dados (chave privada).
# ATENÇÃO: Nunca envie esta chave para o GitHub público!
tmdb.API_KEY = "_______________________"

In [ ]:
# ==============================================================================
# 4. EXTRAÇÃO DE DADOS (EXTRACT)
# ==============================================================================

# Leitura de um arquivo CSV hospedado na web diretamente para a memória como um DataFrame do Pandas.
# Este dataset contém dados gerais de filmes avaliados no site IMDB (Internet Movie Database).
df_imdb_filmes = pd.read_csv(f"{url_github_movies_dataset}/imdb_filmes.csv")

In [ ]:
# Execução de uma consulta SQL estruturada diretamente dentro do Data Warehouse (BigQuery) da nuvem.
# O método '.to_dataframe()' faz o download do resultado da query SQL e o converte em uma tabela Pandas local.
# 'create_bqstorage_client=False' evita criar uma conexão otimizada adicional, mantendo a chamada simples.
df_boxoffice_dc_marvel = bigquery_client.query(
    "SELECT * FROM raw.boxoffice_dc_marvel"
).to_dataframe(create_bqstorage_client=False)

In [ ]:
# ==============================================================================
# 5. SANITIZAÇÃO DE METADADOS (ORDENAÇÃO DE COLUNAS)
# ==============================================================================

# Boas práticas em Engenharia de Dados: Garantir previsibilidade visual.
# Aqui reordenamos as colunas de todas as tabelas em ordem alfabética (A-Z).
# Exemplo: Se as colunas fossem ['B', 'A', 'C'], viram ['A', 'B', 'C'].
df_boxoffice_dc_marvel = df_boxoffice_dc_marvel[sorted(df_boxoffice_dc_marvel.columns)]
df_imdb_filmes = df_imdb_filmes[sorted(df_imdb_filmes.columns)]

 ---
 - Filtrar filmes que pertençam estritamente às franquias "DC" ou "Marvel".

 - Ordenar os filmes pela maior nota (Rating) do IMDB.

 - Capturar quem é o ator principal (o primeiro listado na coluna de "STARS") do Top 10 desses filmes.

 - Buscar informações públicas de postagens e métricas de perfil do Instagram desse ator selecionado.

In [ ]:
# ==============================================================================
# 8. PROCESSAMENTO E CRUZAMENTO DE DADOS (TRANSFORMATION & MERGING)
# ==============================================================================

# Criando uma tabela unificada aplicando conceitos equivalentes ao INNER/OUTER JOIN do SQL:
df_temporario = (
    pd.merge(
        df_boxoffice_dc_marvel,
        df_imdb_filmes,
        left_on=["FILM", "YEAR"],  # Chaves de junção da tabela à esquerda (BoxOffice)
        right_on=["TITLE", "YEAR"],  # Chaves de junção da tabela à direita (IMDB)
        how="outer",  # Tipo de Join: Preserva registros de ambas as tabelas se não houver match direto
    )
    # Ordena cronologicamente do filme mais recente para o mais antigo
    .sort_values(by="YEAR", ascending=False)
    # Remove registros duplicados que possuam exatamente as mesmas informações, mantendo apenas a última ocorrência encontrada
    .drop_duplicates(keep="last")
    # No final do bloco de cruzamento, filtramos (fatiamos) a tabela para manter apenas 4 colunas essenciais
)[["FRANCHISE", "RATING_IMDB", "TITLE", "STAR"]]

In [ ]:
# Filtragem booleana: Seleciona linhas onde a coluna FRANCHISE é igual a 'DC' OU igual a 'Marvel'
# Em seguida, ordena os filmes com base na nota crítica do IMDB de forma decrescente (Maior nota -> Menor nota)
# Isola apenas a coluna "STAR" (que contém os nomes dos atores principais do elenco).
lista_estrelas = df_temporario[
    (df_temporario["FRANCHISE"] == "DC") | (df_temporario["FRANCHISE"] == "Marvel")
].sort_values(by="RATING_IMDB", ascending=False)["STAR"]
lista_estrelas

In [ ]:
# ==============================================================================
# 9. ENGENHARIA DE STRINGS E HIGIENIZAÇÃO DE LISTAS
# ==============================================================================

# A coluna "STAR" vem formatada como uma String única contendo vários atores separados por vírgula.
# Exemplo: "Robert Downey Jr., Chris Evans, Scarlett Johansson"
# Precisamos isolar apenas o primeiro ator de cada filme.
lista_estrelas = (
    lista_estrelas.dropna()  # Remove valores nulos (NaN) para evitar erros de execução.
    .str.strip()  # Remove espaços em branco desnecessários no início e no fim da string.
    .str.split(", ")  # Divide a string em uma lista real do Python baseado na vírgula. Ex: ['Actor1', 'Actor2']
    .str[0]  # Técnica de fatiamento: Pega estritamente o índice 0 (O primeiro ator, considerado o protagonista).
    .explode()  # Transforma elementos de lista em linhas individuais caso houvesse estruturas aninhadas secundárias.
    .str.strip()  # Nova limpeza de segurança contra espaçamentos residuais.
    .unique()  # Remove nomes duplicados, retornando uma lista de atores únicos.
)
lista_estrelas

In [ ]:
# ==============================================================================
# 10. CONSUMO DE API EXTERNA (TMDB - IDENTIFICAÇÃO DE CELEBRIDADES)
# ==============================================================================

# Laço para percorrer apenas os 10 primeiros atores da nossa lista higienizada (Top 10 baseado em melhor avaliação do IMDB).
lista_ids_tmdb_estrelas = []
for i in lista_estrelas[0:10]:
    # Faz uma chamada à API do TMDB buscando textualmente pelo nome do ator ('query=i').
    # A API retorna um dicionário JSON. Buscamos o primeiro resultado '.get("results")[0]' e extraímos o ID numérico único '.get("id")'.
    r = str(tmdb.Search().person(query=i).get("results")[0].get("id"))

    print(f"{i} → {r}")  # Log de console para rastrear qual ID pertence a qual artista.

    # Armazena o ID encontrado na lista de controle.
    lista_ids_tmdb_estrelas.append(r)

print(lista_ids_tmdb_estrelas)

In [ ]:
# ==============================================================================
# 11. ENRIQUECIMENTO DE DADOS (TMDB - CAPTURA DE REDES SOCIAIS)
# ==============================================================================

# Com o ID interno do TMDB, fazemos uma segunda consulta para mapear as redes sociais oficiais cadastradas da pessoa.
lista_instagram_user_estrelas = []
for i in lista_ids_tmdb_estrelas:
    # Método '.external_ids()' retorna chaves como Twitter, Facebook e Instagram.
    r = tmdb.People(i).external_ids().get("instagram_id")

    # Se o ator possuir um usuário de Instagram válido e mapeado, adicionamos à lista.
    if r:
        lista_instagram_user_estrelas.append(r)
        print(r)

In [ ]:
# ==============================================================================
# 12. WEB SCRAPING / EXTRAÇÃO DE DADOS DO INSTAGRAM (POSTS)
# ==============================================================================

# Inicializa o cliente do IG Mapper. Em ambientes produtivos reais, chaves de sessão válidas (cookies) seriam necessárias.
cliente_igmapper = igmapper.InstaClient(csrftoken=None, ds_user_id=None, sessionid=None, use_curl=True)

# Criamos um DataFrame vazio para acumular o histórico consolidado de postagens de todas as celebridades.
df_stars_posts_stats = pd.DataFrame()

for i in lista_instagram_user_estrelas:
    if i:
        print(i)
        # Coleta metadados gerais do perfil (seguidores, publicações totais).
        ig_info = cliente_igmapper.get_profile_info(i)
        print(ig_info.total_posts, ig_info.follower_count, ig_info.follower_count)

        # Coleta o "feed" público recente do usuário. Retorna um dicionário JSON complexo contendo os blocos de posts.
        ig_feed = cliente_igmapper.get_feed(i, max_id=None, return_raw=True)["items"]

        # Converte a lista de posts brutos em uma tabela estruturada temporária.
        df_temporario = pd.DataFrame(ig_feed)

        # Cria uma nova coluna para rastrear e identificar textualmente a quem pertence aquelas publicações.
        df_temporario["user"] = i

        # Concatenação Vertical (UNION ALL do SQL): Empilha a tabela temporária do ator atual no DataFrame consolidado geral.
        df_stars_posts_stats = pd.concat([df_stars_posts_stats, df_temporario], ignore_index=True)
        print()

In [ ]:
# ==============================================================================
# 13. SELEÇÃO DE COLUNAS E REORDENAÇÃO (REQUISITO ESPECÍFICO DO NEGÓCIO)
# ==============================================================================

# Filtra o DataFrame consolidado apenas com os campos solicitados pela área de negócios/análise.
# Ordena o resultado final em ordem alfabética de usuário e, dentro de cada usuário, pelo post mais recente (timestamp decrescente).
df_stars_posts_stats = df_stars_posts_stats[
    ["user", "device_timestamp", "code", "media_type", "carousel_media_count", "like_count", "comment_count"]
].sort_values(by=["user", "device_timestamp"], ascending=False)

In [ ]:
# ==============================================================================
# 14. CARGA DOS POSTS - CAMADA BRONZE (LOAD)
# ==============================================================================

# Exporta os dados refinados das postagens das redes sociais diretamente para a nuvem no BigQuery.
df_stars_posts_stats = df_stars_posts_stats.to_gbq(
    project_id=meu_gcp_project_id,
    destination_table="bronze.stars_posts_stats",
    if_exists="replace",
    credentials=gcp_credentials,
)

In [ ]:
# ==============================================================================
# 15. EXTRAÇÃO DE ESTATÍSTICAS DE PERFIL (PROFILE METRICS)
# ==============================================================================

# Processo similar ao passo 12, mas com foco exclusivo em Métricas Agregadas do Perfil do usuário (e não em postagens individuais).
df_stars_profile_stats = pd.DataFrame()

for i in lista_instagram_user_estrelas:
    if i:
        print(i)
        ig_info = cliente_igmapper.get_profile_info(i)
        print(ig_info.total_posts, ig_info.follower_count, ig_info.follower_count)

        # Em vez de converter um JSON bruto diretamente, mapeamos um dicionário estruturado manualmente com as chaves limpas.
        df_temporario = pd.DataFrame(
            [
                {
                    "user": i,
                    "total_posts": ig_info.total_posts,
                    "followers": ig_info.follower_count,
                    "following": ig_info.following_count,
                }
            ]
        )

        # Faz o empilhamento das métricas desse usuário no DataFrame consolidado de perfis.
        df_stars_profile_stats = pd.concat([df_stars_profile_stats, df_temporario], ignore_index=True)
        print()

In [ ]:
# ==============================================================================
# 16. CARGA FINAL DOS PERFIS - CAMADA BRONZE (LOAD)
# ==============================================================================

# Salva a última tabela analítica no banco de dados em nuvem, completando o ciclo do pipeline de dados.
df_stars_profile_stats = df_stars_profile_stats.to_gbq(
    project_id=meu_gcp_project_id,
    destination_table="bronze.stars_profile_stats",
    if_exists="replace",
    credentials=gcp_credentials,
)